# Analyse sur les accords d'entreprise

Voir ticket : https://github.com/SocialGouv/code-du-travail-numerique/issues/7323

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import pandas as pd
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

In [ ]:
# - 10 juin / now
interval_start = '2026-07-10 00:00:00'
interval_stop = '2026-09-01 00:00:00'

In [ ]:
columns = ['action_id',
    'idvisit',
    'actions',
    'operatingsystemname',
    'action_type',
    'action_timestamp',
    'action_eventcategory',
    'action_eventaction',
    'action_eventname',
    'action_eventvalue',
    'action_url',
    'experiments']

range_query = f"""
        SELECT {", ".join(columns)} FROM matomo_partitioned
        WHERE action_timestamp >= '{interval_start}'
          AND action_timestamp < '{interval_stop}'
        """

In [ ]:
query_visits =  range_query + f"""
          ORDER BY action_timestamp asc;
    """

visits_data = await matomo.run_query(query_visits)

In [ ]:
visits_df = pd.DataFrame(visits_data, columns=columns)

In [ ]:
from urllib.parse import urlsplit, urlunsplit

def clean_url(url):
    if pd.isna(url) or not url:
        return url
    parts = urlsplit(url)
    # on ne garde que scheme + host + path, sans query ni fragment
    cleaned = urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))
    # slash final (mais pas la racine "https://site/")
    if cleaned.endswith("/") and parts.path != "/":
        cleaned = cleaned[:-1]
    return cleaned

visits_df["action_url_clean"] = visits_df["action_url"].apply(clean_url)

## Nb de recherches par entreprises effectuées sur la période juillet / août

### Nb page recherche affiché

In [ ]:
target_url = "https://code.travail.gouv.fr/outils/convention-collective/entreprise"

mask_page = (visits_df["action_type"] == "action") & (visits_df["action_url_clean"] == target_url)
page_views_df = visits_df[mask_page]

print(len(page_views_df), "page views")
print(page_views_df["idvisit"].nunique(), "visites distinctes")

### Nb recherches abouti d'une entreprise

In [ ]:
import re

base_url = "https://code.travail.gouv.fr/outils/convention-collective/entreprise"

# fin d'URL = /<suite de chiffres> uniquement (siren 9 ou siret 14, mais on accepte n'importe quelle longueur)
pattern = rf"^{re.escape(base_url)}/(\d+)$"

mask_siret = (visits_df["action_type"] == "action") & visits_df["action_url_clean"].str.match(pattern, na=False)
siret_page_views_df = visits_df[mask_siret].copy()

# on récupère le siren/siret dans une colonne dédiée
siret_page_views_df["siret"] = siret_page_views_df["action_url_clean"].str.extract(pattern)[0]

print(len(siret_page_views_df), "page views")
print(siret_page_views_df["idvisit"].nunique(), "visites distinctes")

## % des recherches où on a remonté des accords

In [ ]:
mask_show_accords = (
    (visits_df["action_type"] == "event")
    & (visits_df["action_eventcategory"] == "accord_enterprise_search")
    & (visits_df["action_eventaction"] == "show_accords")
)
show_accords_df = visits_df[mask_show_accords]

print(len(show_accords_df), "events show_accords")
print(show_accords_df["idvisit"].nunique(), "visites distinctes avec show_accords")

In [ ]:
show_accords_df = show_accords_df.copy()
show_accords_df["nb_accords"] = pd.to_numeric(show_accords_df["action_eventname"], errors="coerce")

with_accords_df = show_accords_df[show_accords_df["nb_accords"] >= 1]

print(len(with_accords_df), "events show_accords avec >= 1 accord")
print(with_accords_df["idvisit"].nunique(), "visites distinctes avec >= 1 accord")

# et pour comparer avec le total
print(f"→ {len(with_accords_df) / len(show_accords_df):.1%} des events, "
      f"{with_accords_df['idvisit'].nunique() / show_accords_df['idvisit'].nunique():.1%} des visites")

## **% des usagers qui ont cliqué pour consulter leurs accords

In [ ]:
mask_click = (
    (visits_df["action_type"] == "event")
    & (visits_df["action_eventcategory"] == "accord_enterprise_search")
    & (visits_df["action_eventaction"].isin(["click_accord", "click_all_accords"]))
)
click_accords_df = visits_df[mask_click]

# détail par action
click_accords_df.groupby("action_eventaction").agg(
    nb_events=("action_id", "count"),
    nb_visites=("idvisit", "nunique"),
)

# total (visites dédupliquées entre les deux actions)
print(len(click_accords_df), "events click_*")
print(click_accords_df["idvisit"].nunique(), "visites distinctes avec au moins un clic")

In [ ]:
visits_with_accords = set(with_accords_df["idvisit"])
visits_with_click = set(click_accords_df["idvisit"])

print(f"{len(visits_with_click & visits_with_accords)} / {len(visits_with_accords)} visites "
      f"= {len(visits_with_click & visits_with_accords) / len(visits_with_accords):.1%} de clic")

# sanity check : clics sans show_accords >= 1 dans la même visite (devrait être ~0)
print(len(visits_with_click - visits_with_accords), "visites avec clic mais sans show_accords >= 1")